# Problem 2: SynCode

[Open in Google Colab](https://colab.research.google.com/drive/1ElTCNaQRktBsOWI1eBQ2NDvJntS8H7Qh#scrollTo=q2-grammar)

Before running, set the Colab **Runtime Version** to **`2026.07` (Python 3.12)** and select a **T4 GPU**. Reconnect to the runtime, then run all cells.


In [1]:
import sys
if sys.version_info >= (3, 13):
    raise RuntimeError("Select Colab Runtime Version 2026.07 (Python 3.12), then reconnect.")
%pip -q install "syncode==0.4.16" "transformers==4.53.2" "accelerate>=1,<2"

In [2]:
import math
import operator
import re
from functools import lru_cache
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
from syncode import Grammar, SyncodeLogitsProcessor

MODEL_ID = "microsoft/phi-2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

In [3]:
NUMBER_PATTERN = r"[+-]?(?:[0-9]+(?:\.[0-9]*)?|\.[0-9]+)"
GRAMMAR = r"""
start: EXPR
EXPR: / ?\([+-]?([0-9]+(\.[0-9]*)?|\.[0-9]+) ?[+*\/-] ?[+-]?([0-9]+(\.[0-9]*)?|\.[0-9]+)\)/
"""
RECORD_RE = re.compile(
    rf"\s*\(\s*({NUMBER_PATTERN})\s*([+*/-])\s*({NUMBER_PATTERN})\s*\)\s*"
)
OPERATIONS = {"+": operator.add, "-": operator.sub,
              "*": operator.mul, "/": operator.truediv}

PROMPT_PREFIX = """Extract the arithmetic expression from each question. Copy both operands with their signs.
Do not calculate the answer. Preserve operand order: a minus b means a - b;
subtract a from b means b - a. Output only: (left operator right).

Question: What is -6.25 plus 2.5?
Expression: (-6.25 + 2.5)

Question: What is 4.5 minus 9.25?
Expression: (4.5 - 9.25)

Question: Subtract -1.5 from 6.0.
Expression: (6.0 - -1.5)

Question: What is -7.2 multiplied by -3.0?
Expression: (-7.2 * -3.0)

"""

def parse_and_calculate(record):
    match = RECORD_RE.fullmatch(record)
    if match is None:
        raise ValueError(f"Incomplete or invalid extraction: {record!r}")
    left, right = float(match[1]), float(match[3])
    op = match[2]
    if not all(math.isfinite(v) for v in (left, right)):
        raise ValueError("Operands must be finite.")
    if op == "/" and right == 0:
        raise ZeroDivisionError("Division by zero.")
    answer = float(OPERATIONS[op](left, right))
    if not math.isfinite(answer):
        raise ValueError("The result is not finite.")
    return ([left, right], op, answer)

In [4]:
@lru_cache(maxsize=1)
def load_extractor():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, low_cpu_mem_usage=True,
        attn_implementation="eager",
    ).to(DEVICE).eval()
    processor = SyncodeLogitsProcessor(
        grammar=Grammar(GRAMMAR), tokenizer=tokenizer,
        parse_output_only=True, mode="grammar_strict",
    )
    return model, tokenizer, processor


class StopAtCompleteRecord(StoppingCriteria):
    def __init__(self, tokenizer, prompt_length):
        self.tokenizer, self.prompt_length = tokenizer, prompt_length

    def __call__(self, input_ids, scores, **kwargs):
        text = self.tokenizer.decode(input_ids[0, self.prompt_length:], skip_special_tokens=True)
        return RECORD_RE.fullmatch(text) is not None


@torch.inference_mode()
def run_syncode(textual_query):
    if not isinstance(textual_query, str) or not textual_query.strip():
        raise ValueError("textual_query must be a nonempty string.")
    model, tokenizer, processor = load_extractor()
    prompt = PROMPT_PREFIX + "Question: " + textual_query.strip() + "\nExpression:"
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    n = inputs.input_ids.shape[1]
    if n + 80 > model.config.max_position_embeddings:
        raise ValueError("Query exceeds the model context budget.")
    processor.reset()
    output = model.generate(
        **inputs, max_new_tokens=80, do_sample=False, num_beams=1,
        use_cache=True, logits_processor=[processor],
        stopping_criteria=StoppingCriteriaList([StopAtCompleteRecord(tokenizer, n)]),
        pad_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id,
    )
    record = tokenizer.decode(output[0, n:], skip_special_tokens=True).strip()
    return parse_and_calculate(record)

In [5]:
# Optional self-checks; not required by run_syncode.
TEST_CASES = [
    ('What is 327. multiplied by 11.0?', [327.0, 11.0], '*', 3597.0),
    ('What is 45.1 plus 23.54?', [45.1, 23.54], '+', 68.64),
    ('What is 120.4 divided by 4.0?', [120.4, 4.0], '/', 30.1),
    ('What is 15.75 minus 4.25?', [15.75, 4.25], '-', 11.5),
    ('Subtract 7.5 from 20.0.', [20.0, 7.5], '-', 12.5),
    ('What is -8.5 plus 2.25?', [-8.5, 2.25], '+', -6.25),
    ('What is -6.0 multiplied by -2.5?', [-6.0, -2.5], '*', 15.0),
    ('What is 7.5 divided by -2.5?', [7.5, -2.5], '/', -3.0),
    ('What is .75 plus .125?', [0.75, 0.125], '+', 0.875),
    ('What is 12. multiplied by 0.0?', [12.0, 0.0], '*', 0.0),
    ('What is 0.0 divided by 5.5?', [0.0, 5.5], '/', 0.0),
    ('What is 3.0 minus 8.0?', [3.0, 8.0], '-', -5.0),
    ('What is 1000.25 plus 0.75?', [1000.25, 0.75], '+', 1001.0),
    ('Subtract -2.5 from 4.0.', [4.0, -2.5], '-', 6.5),
    ('What is 0.1 plus 0.2?', [0.1, 0.2], '+', 0.3),
]
passed = 0
for query, operands, op, answer in TEST_CASES:
    try:
        result = run_syncode(query)
        ok = (result[:2] == (operands, op)
              and math.isclose(result[2], answer, rel_tol=1e-9, abs_tol=1e-9))
    except Exception as exc:
        result, ok = f"{type(exc).__name__}: {exc}", False
    passed += ok
    print("PASS" if ok else "FAIL", query, "->", result)
print(f"{passed}/{len(TEST_CASES)} passed")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


PASS What is 327. multiplied by 11.0? -> ([327.0, 11.0], '*', 3597.0)
PASS What is 45.1 plus 23.54? -> ([45.1, 23.54], '+', 68.64)
PASS What is 120.4 divided by 4.0? -> ([120.4, 4.0], '/', 30.1)
PASS What is 15.75 minus 4.25? -> ([15.75, 4.25], '-', 11.5)
PASS Subtract 7.5 from 20.0. -> ([20.0, 7.5], '-', 12.5)
PASS What is -8.5 plus 2.25? -> ([-8.5, 2.25], '+', -6.25)
PASS What is -6.0 multiplied by -2.5? -> ([-6.0, -2.5], '*', 15.0)
PASS What is 7.5 divided by -2.5? -> ([7.5, -2.5], '/', -3.0)
PASS What is .75 plus .125? -> ([0.75, 0.125], '+', 0.875)
PASS What is 12. multiplied by 0.0? -> ([12.0, 0.0], '*', 0.0)
PASS What is 0.0 divided by 5.5? -> ([0.0, 5.5], '/', 0.0)
PASS What is 3.0 minus 8.0? -> ([3.0, 8.0], '-', -5.0)
PASS What is 1000.25 plus 0.75? -> ([1000.25, 0.75], '+', 1001.0)
PASS Subtract -2.5 from 4.0. -> ([4.0, -2.5], '-', 6.5)
PASS What is 0.1 plus 0.2? -> ([0.1, 0.2], '+', 0.30000000000000004)
15/15 passed
